# Experimento: Gradiente Evolutivo SW → HW → Grass en t_gen

**Hipótesis:** Si el gradiente evolutivo SW→HW→Grass se refleja en t_gen, entonces:
> t_gen(SW) > t_gen(HW) > t_gen(GR) para DP equivalente

**Diseño:** 3 grupos × 3 niveles de DP × 30 réplicas = 270 corridas  
**Output:** CSV con t_gen individual por réplica, listo para análisis estadístico

## Instrucciones
1. Copiar los 9 YAMLs del experimento a la carpeta `resources/`
2. Verificar rutas en CELL 1 (BASE_PATH)
3. Ejecutar todas las celdas en orden
4. El CSV se guarda incrementalmente — si se interrumpe, se puede retomar

In [1]:
# ============================================================
# CELL 1: CONFIGURACIÓN — AJUSTAR ANTES DE CORRER
# ============================================================

import os
import subprocess
import shutil
import time
import glob
import csv
from datetime import datetime

# --- AJUSTAR ESTAS RUTAS ---
BASE_PATH   = "/media/aldo/Aldo/sims/Lignin_DB_Natural_Evol"  # Igual que el notebook original
YAML_DIR    = os.path.join(BASE_PATH, "resources")                  # Donde están los 9 YAMLs del experimento
OUTPUT_CSV  = os.path.join(BASE_PATH, "output", "evolutionary_experiment_results.csv")
N_REPLICAS  = 30   # Réplicas por configuración
# ---------------------------

# Patrón de los YAMLs del experimento (SW/HW/GR únicamente)
EXPERIMENT_PATTERN = "[SHG][WR]*_DP*_config.yaml"  # Captura SW_, HW_, GR_

JAR_PATH       = os.path.join(BASE_PATH, "lgs_gen.jar")
PROJECT_CONFIG = os.path.join(YAML_DIR, "project_config.yaml")
OUTPUT_DIR     = os.path.join(BASE_PATH, "output")

# Verificar que el jar existe
if not os.path.exists(JAR_PATH):
    raise FileNotFoundError(f"❌ lgs_gen.jar no encontrado en: {JAR_PATH}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ lgs_gen.jar encontrado")
print(f"📁 YAML dir:    {YAML_DIR}")
print(f"📊 Output CSV:  {OUTPUT_CSV}")
print(f"🔁 Réplicas:    {N_REPLICAS} por configuración")

✅ lgs_gen.jar encontrado
📁 YAML dir:    /media/aldo/Aldo/sims/Lignin_DB_Natural_Evol/resources
📊 Output CSV:  /media/aldo/Aldo/sims/Lignin_DB_Natural_Evol/output/evolutionary_experiment_results.csv
🔁 Réplicas:    30 por configuración


In [3]:
# ============================================================
# CELL 2: ENCONTRAR Y VERIFICAR LOS 9 YAMLs DEL EXPERIMENTO
# ============================================================

# Buscar exactamente los YAMLs del experimento evolutivo
all_yamls = sorted(glob.glob(os.path.join(YAML_DIR, "*_config.yaml")))

# Filtrar solo los grupos experimentales SW, HW, GR
experiment_yamls = [
    f for f in all_yamls 
    if any(os.path.basename(f).startswith(g) for g in ["SW_", "HW_", "GR_"])
]

print(f"🔍 YAMLs del experimento encontrados: {len(experiment_yamls)}")
print()

# Tabla de verificación
print(f"{'Archivo':<30} {'Grupo':<6} {'DP':<5} {'fBO4 esperado'}")
print("-" * 60)

expected = {
    "SW": {"fBO4": 0.54, "sg": 0.0},
    "HW": {"fBO4": 0.69, "sg": 1.5},
    "GR": {"fBO4": 0.77, "sg": 0.8},
}

for f in experiment_yamls:
    name = os.path.basename(f).replace("_config.yaml", "")
    parts = name.split("_")
    group = parts[0]
    dp = parts[1].replace("DP", "")
    fBO4 = expected.get(group, {}).get("fBO4", "?")
    print(f"{name:<30} {group:<6} {dp:<5} {fBO4}")

if len(experiment_yamls) != 9:
    print(f"\n⚠️  Se esperaban 9 YAMLs, se encontraron {len(experiment_yamls)}")
    print("   Verificar que los archivos SW_DPx, HW_DPx, GR_DPx están en:", YAML_DIR)
else:
    print(f"\n✅ Los 9 archivos están presentes")
    
# Estimación de tiempo total
print()
print("⏱️  ESTIMACIÓN DE TIEMPO (basada en corridas previas):")
print("   DP5:  ~2s/corrida  × 30 × 3 grupos = ~3 min")
print("   DP8:  ~15s/corrida × 30 × 3 grupos = ~23 min")
print("   DP10: ~80s/corrida × 30 × 3 grupos = ~120 min")
print("   TOTAL ESTIMADO: ~2.5–4 horas")
print("   (El CSV se guarda después de cada réplica — se puede interrumpir)")

🔍 YAMLs del experimento encontrados: 9

Archivo                        Grupo  DP    fBO4 esperado
------------------------------------------------------------
GR_DP10                        GR     10    0.77
GR_DP5                         GR     5     0.77
GR_DP8                         GR     8     0.77
HW_DP10                        HW     10    0.69
HW_DP5                         HW     5     0.69
HW_DP8                         HW     8     0.69
SW_DP10                        SW     10    0.54
SW_DP5                         SW     5     0.54
SW_DP8                         SW     8     0.54

✅ Los 9 archivos están presentes

⏱️  ESTIMACIÓN DE TIEMPO (basada en corridas previas):
   DP5:  ~2s/corrida  × 30 × 3 grupos = ~3 min
   DP8:  ~15s/corrida × 30 × 3 grupos = ~23 min
   DP10: ~80s/corrida × 30 × 3 grupos = ~120 min
   TOTAL ESTIMADO: ~2.5–4 horas
   (El CSV se guarda después de cada réplica — se puede interrumpir)


In [4]:
# ============================================================
# CELL 3: CLASE RUNNER CON SOPORTE DE RÉPLICAS Y CHECKPOINT
# ============================================================

class EvolutionaryExperimentRunner:
    """
    Corre el experimento evolutivo con n réplicas por configuración.
    Guarda resultados incrementalmente en CSV (checkpoint).
    Si se interrumpe, retoma desde donde quedó.
    """
    
    def __init__(self, base_path, output_csv, n_replicas=30):
        self.base_path      = base_path
        self.jar_path       = os.path.join(base_path, "lgs_gen.jar")
        self.resources_dir  = os.path.join(base_path, "resources")
        self.output_dir     = os.path.join(base_path, "output")
        self.output_csv     = output_csv
        self.n_replicas     = n_replicas
        
        # Columnas del CSV
        self.csv_columns = [
            "config_name", "group", "dp", "replica",
            "t_gen_s", "success", "timestamp"
        ]
        
        # Cargar corridas ya completadas (para retomar si se interrumpe)
        self.completed = self._load_completed()
        if self.completed:
            print(f"♻️  Retomando: {len(self.completed)} corridas ya completadas")
        
        # Inicializar CSV si no existe
        if not os.path.exists(self.output_csv):
            with open(self.output_csv, "w", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=self.csv_columns)
                writer.writeheader()
    
    def _load_completed(self):
        """Lee el CSV existente y devuelve set de (config_name, replica) ya hechas."""
        completed = set()
        if os.path.exists(self.output_csv):
            with open(self.output_csv, "r") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    completed.add((row["config_name"], int(row["replica"])))
        return completed
    
    def _parse_config_name(self, config_file):
        """Extrae group y dp del nombre de archivo."""
        name = os.path.basename(config_file).replace("_config.yaml", "")
        parts = name.split("_")
        group = parts[0]          # SW, HW, GR
        dp    = int(parts[1].replace("DP", ""))  # 5, 8, 10
        return name, group, dp
    
    def _cleanup_lgs_outputs(self):
        """Limpia carpetas de output de LGS para que no interfieran con la próxima corrida."""
        for folder in ["json", "mol", "struct", "matrices"]:
            path = os.path.join(self.output_dir, folder)
            if os.path.exists(path):
                shutil.rmtree(path)
        # También limpiar cualquier output renombrado de réplicas anteriores
        for item in glob.glob(os.path.join(self.output_dir, "*_json")):
            shutil.rmtree(item, ignore_errors=True)
        for item in glob.glob(os.path.join(self.output_dir, "*_mol")):
            shutil.rmtree(item, ignore_errors=True)
        for item in glob.glob(os.path.join(self.output_dir, "*_struct")):
            shutil.rmtree(item, ignore_errors=True)
        for item in glob.glob(os.path.join(self.output_dir, "*_matrices")):
            shutil.rmtree(item, ignore_errors=True)
    
    def _run_once(self, config_file, config_name):
        """Ejecuta LGS una sola vez con el YAML dado. Devuelve t_gen en segundos."""
        
        project_config = os.path.join(self.resources_dir, "project_config.yaml")
        backup = None
        
        # Backup del project_config si existe
        if os.path.exists(project_config):
            backup = project_config + ".backup"
            shutil.move(project_config, backup)
        
        try:
            # Limpiar outputs anteriores
            self._cleanup_lgs_outputs()
            
            # Copiar el YAML experimental como project_config
            shutil.copy2(config_file, project_config)
            
            # Correr LGS y medir tiempo
            start = time.time()
            result = subprocess.run(
                ["java", "-jar", self.jar_path],
                capture_output=True,
                text=True,
                cwd=self.base_path
            )
            t_gen = time.time() - start
            
            success = result.returncode == 0
            return t_gen, success
        
        finally:
            # Restaurar project_config original
            if os.path.exists(project_config):
                os.remove(project_config)
            if backup and os.path.exists(backup):
                shutil.move(backup, project_config)
    
    def _save_result(self, config_name, group, dp, replica, t_gen, success):
        """Agrega una fila al CSV (modo append — checkpoint incremental)."""
        row = {
            "config_name": config_name,
            "group":       group,
            "dp":          dp,
            "replica":     replica,
            "t_gen_s":     round(t_gen, 3),
            "success":     success,
            "timestamp":   datetime.now().isoformat()
        }
        with open(self.output_csv, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.csv_columns)
            writer.writerow(row)
    
    def run_experiment(self, config_files):
        """Corre el experimento completo: n réplicas × cada config."""
        
        total_runs = len(config_files) * self.n_replicas
        runs_done  = len(self.completed)
        runs_left  = total_runs - runs_done
        
        print(f"🧬 EXPERIMENTO EVOLUTIVO")
        print(f"   Configuraciones: {len(config_files)}")
        print(f"   Réplicas/config: {self.n_replicas}")
        print(f"   Total corridas:  {total_runs}")
        print(f"   Ya completadas:  {runs_done}")
        print(f"   Por correr:      {runs_left}")
        print(f"   CSV output:      {self.output_csv}")
        print("-" * 60)
        
        all_results = []
        
        for config_file in config_files:
            config_name, group, dp = self._parse_config_name(config_file)
            
            print(f"\n📂 {config_name}  [grupo={group}, DP={dp}]")
            
            t_gens_this_config = []
            
            for rep in range(1, self.n_replicas + 1):
                
                # Skip si ya está completada
                if (config_name, rep) in self.completed:
                    print(f"   [{rep:02d}/{self.n_replicas}] ⏭️  Ya completada")
                    continue
                
                # Correr LGS
                t_gen, success = self._run_once(config_file, config_name)
                t_gens_this_config.append(t_gen)
                
                # Guardar al CSV inmediatamente (checkpoint)
                self._save_result(config_name, group, dp, rep, t_gen, success)
                self.completed.add((config_name, rep))
                
                status = "✅" if success else "❌"
                print(f"   [{rep:02d}/{self.n_replicas}] {status} t_gen = {t_gen:.2f}s")
                
                all_results.append({
                    "config_name": config_name,
                    "group": group,
                    "dp": dp,
                    "replica": rep,
                    "t_gen_s": t_gen,
                    "success": success
                })
            
            # Resumen de la config
            if t_gens_this_config:
                mean_t = sum(t_gens_this_config) / len(t_gens_this_config)
                min_t  = min(t_gens_this_config)
                max_t  = max(t_gens_this_config)
                print(f"   → Media: {mean_t:.2f}s | Min: {min_t:.2f}s | Max: {max_t:.2f}s")
        
        print(f"\n{'='*60}")
        print(f"✅ EXPERIMENTO COMPLETADO")
        print(f"   Resultados guardados en: {self.output_csv}")
        
        return all_results

print("✅ Clase EvolutionaryExperimentRunner definida")

✅ Clase EvolutionaryExperimentRunner definida


In [5]:
# ============================================================
# CELL 4: EJECUTAR EL EXPERIMENTO
# ============================================================
# ⚠️  Esta celda puede tardar 2-4 horas.
# El CSV se guarda después de cada réplica.
# Si se interrumpe, volver a correr esta celda — retomará automáticamente.

runner = EvolutionaryExperimentRunner(
    base_path   = BASE_PATH,
    output_csv  = OUTPUT_CSV,
    n_replicas  = N_REPLICAS
)

results = runner.run_experiment(experiment_yamls)

🧬 EXPERIMENTO EVOLUTIVO
   Configuraciones: 9
   Réplicas/config: 30
   Total corridas:  270
   Ya completadas:  0
   Por correr:      270
   CSV output:      /media/aldo/Aldo/sims/Lignin_DB_Natural_Evol/output/evolutionary_experiment_results.csv
------------------------------------------------------------

📂 GR_DP10  [grupo=GR, DP=10]
   [01/30] ✅ t_gen = 39.26s
   [02/30] ✅ t_gen = 37.52s
   [03/30] ✅ t_gen = 40.24s
   [04/30] ✅ t_gen = 37.42s
   [05/30] ✅ t_gen = 40.66s
   [06/30] ✅ t_gen = 36.65s
   [07/30] ✅ t_gen = 38.16s
   [08/30] ✅ t_gen = 39.01s
   [09/30] ✅ t_gen = 34.60s
   [10/30] ✅ t_gen = 35.72s
   [11/30] ✅ t_gen = 34.87s
   [12/30] ✅ t_gen = 34.86s
   [13/30] ✅ t_gen = 35.55s
   [14/30] ✅ t_gen = 38.68s
   [15/30] ✅ t_gen = 35.58s
   [16/30] ✅ t_gen = 34.75s
   [17/30] ✅ t_gen = 37.40s
   [18/30] ✅ t_gen = 35.78s
   [19/30] ✅ t_gen = 33.95s
   [20/30] ✅ t_gen = 38.85s
   [21/30] ✅ t_gen = 38.33s
   [22/30] ✅ t_gen = 38.70s
   [23/30] ✅ t_gen = 33.37s
   [24/30] ✅ t_gen

In [6]:
# ============================================================
# CELL 5: VERIFICACIÓN RÁPIDA DEL CSV GENERADO
# ============================================================
# Correr esta celda después de que termine (o en cualquier momento
# para ver el progreso parcial)

import csv
from collections import defaultdict

# Leer CSV
rows = []
with open(OUTPUT_CSV, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

print(f"📊 Total de corridas registradas: {len(rows)}")
print(f"   Esperadas: {9 * N_REPLICAS}")
print()

# Resumen por grupo y DP
from collections import defaultdict
import statistics

summary = defaultdict(list)
for row in rows:
    if row["success"] == "True":
        key = (row["group"], int(row["dp"]))
        summary[key].append(float(row["t_gen_s"]))

print(f"{'Grupo':<8} {'DP':<5} {'N':>5} {'Media (s)':>12} {'SD (s)':>10} {'Min':>8} {'Max':>8}")
print("-" * 60)

for (group, dp) in sorted(summary.keys()):
    vals = summary[(group, dp)]
    if len(vals) > 1:
        print(f"{group:<8} {dp:<5} {len(vals):>5} {statistics.mean(vals):>12.2f} "
              f"{statistics.stdev(vals):>10.2f} {min(vals):>8.2f} {max(vals):>8.2f}")
    elif len(vals) == 1:
        print(f"{group:<8} {dp:<5} {len(vals):>5} {vals[0]:>12.2f} {'N/A':>10} {vals[0]:>8.2f} {vals[0]:>8.2f}")

print()
print("✅ Si el gradiente se cumple: t_gen(SW) > t_gen(HW) > t_gen(GR) en cada DP")

📊 Total de corridas registradas: 270
   Esperadas: 270

Grupo    DP        N    Media (s)     SD (s)      Min      Max
------------------------------------------------------------
GR       5        30         4.90       0.27     4.53     5.67
GR       8        30        17.00       1.43    14.85    21.34
GR       10       30        37.03       2.25    33.37    43.32
HW       5        30         3.95       0.25     3.61     4.54
HW       8        30        17.94       2.05    14.40    22.37
HW       10       30        68.12       3.30    62.43    75.53
SW       5        30         3.37       0.21     3.04     3.99
SW       8        30        26.09       2.13    23.36    30.75
SW       10       30        21.86       1.12    19.99    23.80

✅ Si el gradiente se cumple: t_gen(SW) > t_gen(HW) > t_gen(GR) en cada DP
